# CIFAR-10: shared baseline (20 epochs)
This notebook starts with the original SimpleCNN, normalization-only preprocessing and SGD with momentum. It trains one baseline run, checks validation after each epoch, and evaluates the official test set once after selecting the best validation epoch. The same split and pipeline can be reused for the architecture, augmentation and optimizer studies.

Run all cells from top to bottom. Training needs the CIFAR-10 download and takes longer on CPU.

In [ ]:
import copy
import csv
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision.models import densenet121, resnet18
from torchvision.transforms import v2


## 1. Fixed settings
The run seed is **0** and the split seed is **42**, as in the original notebook. Keep the split seed fixed for *every* later configuration and use the same three run seeds (0, 1, 2) for each configuration when expanding the study. A fixed 20-epoch budget, batch size, input size, normalization, loss and learning-rate policy make the comparisons easier to interpret.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('cifar10_data')
RESULTS_DIR = Path('results')
SPLIT_SEED = 42
RUN_SEED = 0
EPOCHS = 20
BATCH_SIZE = 128
CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

# C1 is the shared baseline: future factor studies should change one key at a time.
baseline = {
    'id': 'C1',
    'architecture': 'simple_cnn',
    'augmentation': 'none',
    'optimizer': 'sgd_momentum',
    'learning_rate': 0.01,
}
print('Device:', DEVICE, '| Run seed:', RUN_SEED, '| Split seed:', SPLIT_SEED)


## 2. Architectures
All models start with random weights and accept 32 × 32 images. The ResNet and DenseNet stems are adapted to this image size. Only SimpleCNN is trained in this baseline run.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def get_cifar_resnet18():
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def get_cifar_densenet121():
    model = densenet121(weights=None, num_classes=10)
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()
    return model


architectures = {
    'simple_cnn': SimpleCNN,
    'resnet18': get_cifar_resnet18,
    'densenet121': get_cifar_densenet121,
}


## 3. Preprocessing and optimizers
Augmentation is for training only. Validation and test always use the same deterministic transform. Learning rates are explicit; the baseline uses a constant learning rate for all 20 epochs.

In [ ]:
normalization = [
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
]
eval_transform = v2.Compose([v2.ToImage(), *normalization])
augmentation_strategies = {
    'none': eval_transform,
    'crop_flip': v2.Compose([
        v2.ToImage(), v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
        *normalization,
    ]),
    'crop_flip_randaugment': v2.Compose([
        v2.ToImage(), v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
        v2.RandAugment(num_ops=2, magnitude=9), *normalization,
    ]),
}

def build_optimizer(name, parameters, learning_rate):
    if name == 'sgd_momentum':
        return optim.SGD(parameters, lr=learning_rate, momentum=0.9)
    if name == 'adam':
        return optim.Adam(parameters, lr=learning_rate)
    if name == 'adamw':
        return optim.AdamW(parameters, lr=learning_rate, weight_decay=0.01)
    raise ValueError(f'Unknown optimizer: {name}')


## 4. Fixed stratified split and data loaders
Split only the official 50,000 training images; the 10,000 test images remain untouched. The same 45,000/5,000 split is reused for later experiments. Separate CIFAR-10 objects let the training and validation subsets have different transforms without changing the underlying images.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
raw_train = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=True)
train_idx, val_idx = train_test_split(
    np.arange(len(raw_train)), test_size=5000,
    stratify=raw_train.targets, random_state=SPLIT_SEED,
)
assert len(train_idx) == 45000 and len(val_idx) == 5000
assert len(set(train_idx) & set(val_idx)) == 0
print('Training images:', len(train_idx), '| Validation images:', len(val_idx))


def make_loaders(augmentation, seed):
    train_data = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=False,
        transform=augmentation_strategies[augmentation],
    )
    val_data = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=False, transform=eval_transform,
    )
    test_data = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=False, download=True, transform=eval_transform,
    )
    shuffle_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(Subset(train_data, train_idx), batch_size=BATCH_SIZE,
                              shuffle=True, generator=shuffle_generator, num_workers=0)
    val_loader = DataLoader(Subset(val_data, val_idx), batch_size=BATCH_SIZE, num_workers=0)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=0)
    return train_loader, val_loader, test_loader


## 5. Train and evaluate
One shared training function keeps future comparisons consistent. Each epoch logs sample-weighted loss and accuracy for both training and validation. The test loader is used only after the best epoch has been chosen from validation accuracy.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, collect_predictions=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    true_labels, predictions = [], []
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            predicted = outputs.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            if collect_predictions:
                true_labels.extend(labels.cpu().tolist())
                predictions.extend(predicted.cpu().tolist())
    return total_loss / total, correct / total, true_labels, predictions


def run_experiment(config, seed):
    set_seed(seed)  # Reset initialization, shuffling and augmentation for every run.
    train_loader, val_loader, test_loader = make_loaders(config['augmentation'], seed)
    model = architectures[config['architecture']]().to(DEVICE)
    optimizer = build_optimizer(config['optimizer'], model.parameters(), config['learning_rate'])
    criterion = nn.CrossEntropyLoss()
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    history = []
    best_accuracy, best_weights, best_epoch = -1.0, None, None
    start_time = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_accuracy = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_accuracy, _, _ = evaluate(model, val_loader, criterion)
        history.append({
            'configuration_id': config['id'], 'seed': seed, 'epoch': epoch,
            'train_loss': train_loss, 'train_accuracy': train_accuracy,
            'val_loss': val_loss, 'val_accuracy': val_accuracy,
        })
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_weights = copy.deepcopy(model.state_dict())
            best_epoch = epoch
        print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f}, '
              f'acc {train_accuracy:.3f} | val loss {val_loss:.4f}, acc {val_accuracy:.3f}')

    training_seconds = time.perf_counter() - start_time
    model.load_state_dict(best_weights)
    _, test_accuracy, true_labels, predictions = evaluate(
        model, test_loader, criterion, collect_predictions=True,
    )
    labels = list(range(len(CLASSES)))
    precision, recall, _, _ = precision_recall_fscore_support(
        true_labels, predictions, labels=labels, zero_division=0,
    )
    per_class = [
        {'configuration_id': config['id'], 'seed': seed, 'class': name,
         'precision': float(precision[i]), 'recall': float(recall[i])}
        for i, name in enumerate(CLASSES)
    ]
    result = {
        'configuration_id': config['id'], 'seed': seed,
        'architecture': config['architecture'], 'augmentation': config['augmentation'],
        'optimizer': config['optimizer'], 'learning_rate': config['learning_rate'],
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'split_seed': SPLIT_SEED,
        'best_epoch': best_epoch, 'parameter_count': parameter_count,
        'training_seconds': training_seconds,
        'test_accuracy': accuracy_score(true_labels, predictions),
        'test_macro_f1': f1_score(true_labels, predictions, average='macro'),
    }
    assert abs(result['test_accuracy'] - test_accuracy) < 1e-12
    return result, history, per_class, confusion_matrix(true_labels, predictions, labels=labels)


## 6. Run the baseline and save results
The CSV files are generated from this run; they are not fabricated example results. Add more configurations and seeds later using the same `run_experiment` function. Do not submit downloaded CIFAR-10 images or model checkpoints.

In [ ]:
result, history, per_class, matrix = run_experiment(baseline, RUN_SEED)
RESULTS_DIR.mkdir(exist_ok=True)

def save_rows(path, rows):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

save_rows(RESULTS_DIR / 'results.csv', [result])
save_rows(RESULTS_DIR / 'history.csv', history)
save_rows(RESULTS_DIR / 'per_class.csv', per_class)
with (RESULTS_DIR / 'confusion_matrix.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['actual / predicted', *CLASSES])
    for name, row in zip(CLASSES, matrix):
        writer.writerow([name, *row])

print(f"Best validation epoch: {result['best_epoch']}")
print(f"Test accuracy: {result['test_accuracy']:.3f}")
print(f"Test macro-F1: {result['test_macro_f1']:.3f}")
print(f'Results saved to {RESULTS_DIR.resolve()}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, label in [('train_loss', 'Train'), ('val_loss', 'Validation')]:
    axes[0].plot([row['epoch'] for row in history], [row[key] for row in history], label=label)
for key, label in [('train_accuracy', 'Train'), ('val_accuracy', 'Validation')]:
    axes[1].plot([row['epoch'] for row in history], [row[key] for row in history], label=label)
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy loss')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Fraction correct')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
